# BayMorph — Multi-Observation Joint Fit

This notebook demonstrates joint Bayesian inference from multiple
user-supplied observations of the same astronomical object.

BayMorph does not distribute observational data with the source repository or
installed package. Replace the example paths, target seeds, and scientific
priors below before running the workflow.

The observations remain distinct measurements on their native detector grids.
Parameter sharing is an explicit modelling choice and is never inferred from
observation names.

The labels `g`, `r`, and `i` are illustrative observation identifiers. They
may represent filters, epochs, instruments, or any other distinct observations.


## 1. Define the observations

FITS paths and target-identification seeds are explicit user inputs.

Replace each path with the corresponding science FITS file. Replace each
`None` target centre with a zero-based `(x, y)` pixel coordinate identifying
the intended source in that observation. These coordinates seed preparation;
they are not fixed fitted centroids.

The flux-prior bounds are also observation-specific scientific inputs. They
must be declared or calibrated for the supplied data rather than copied from
another target.


In [ ]:
from pathlib import Path

from baymorph import (
    BayMorphRunner,
    make_multi_observation_config,
    make_multi_parameters,
)


observations = {
    "g": Path("data/observed_galaxy_g.fits"),
    "r": Path("data/observed_galaxy_r.fits"),
    "i": Path("data/observed_galaxy_i.fits"),
}

target_centers = {
    "g": None,
    "r": None,
    "i": None,
}

flux_prior_bounds = {
    "g": None,
    "r": None,
    "i": None,
}

analysis_name = "observed_galaxy_gri_single_sersic"
model_name = "single_sersic"


## 2. Define the scientific coupling

Parameter coupling is part of the probabilistic model.

This template shares axis ratio and position angle as an explicit example
hypothesis. Flux, effective radius, Sérsic index, sky position, and sky
background remain observation-specific.

Sharing sky position through `ra` and `dec` is appropriate only after the WCS
registration and astrometric assumptions have been validated. Sharing
effective radius or Sérsic index is another stronger scientific hypothesis and
should be tested deliberately.

The non-flux bounds below are illustrative starting choices, not universal
BayMorph defaults. Review every prior against the supplied observations and
the scientific question.


In [ ]:
missing_centres = [
    name
    for name, centre in target_centers.items()
    if centre is None
]

if missing_centres:
    raise ValueError(
        "Set an explicit target centre for every observation: "
        f"{missing_centres}."
    )


missing_flux_priors = [
    name
    for name, bounds in flux_prior_bounds.items()
    if bounds is None
]

if missing_flux_priors:
    raise ValueError(
        "Set or calibrate flux-prior bounds for every observation: "
        f"{missing_flux_priors}."
    )


prior_bounds = {
    "flux": flux_prior_bounds,
    "re_arcsec": (0.2, 45.0),
    "n": (0.2, 6.0),
    "q": (0.1, 1.0),
}

shared_parameters = (
    "q",
    "position_angle",
)

parameters = make_multi_parameters(
    model=model_name,
    observations=observations,
    target_centers=target_centers,
    prior_bounds=prior_bounds,
    shared=shared_parameters,
    parameter_frame="sky",
)


## 3. Generate the joint configuration

BayMorph writes one configuration for each observation and one joint
configuration describing the coupled inference problem.

This generic template uses a normalized Gaussian likelihood for every
observation. Per-observation likelihood overrides may be supplied when
different, independently validated noise models are required.

Nautilus is the default production backend. Here its numerical settings are
made explicit only for reproducibility; no backend-specific Python machinery
is required by the notebook.


In [ ]:
config_dir = Path("runs") / analysis_name / "configs"
joint_config_path = config_dir / "joint.yaml"

joint_config = make_multi_observation_config(
    observations=observations,
    target_positions=target_centers,
    parameters=parameters,
    output_dir=config_dir,
    joint_output_path=joint_config_path,
    analysis_name=analysis_name,
    model_name=model_name,
    parameter_frame="sky",
    likelihood={
        "type": "gaussian",
        "include_normalization": True,
    },
    sampler={
        "name": "nautilus",
        "init": {
            "n_live": 300,
            "seed": 42,
        },
        "run": {
            "n_eff": 500,
            "f_live": 0.02,
        },
    },
    outputs_root="runs",
)

print(f"Joint configuration: {joint_config_path}")

for name, entry in joint_config["observations"].items():
    print(f"{name}: {entry['config_path']}")


## 4. All-in-one workflow

`BayMorphRunner` automatically recognizes the joint configuration.

The same high-level interface used for a single observation coordinates
preparation, PSF construction, joint inference, persistence, and
diagnostics.

In [ ]:
runner = BayMorphRunner(
    joint_config_path,
    verbose=True,
)

result = runner.run(
    diagnostics=True,
)

result

## 5. Step-by-step equivalent

The all-in-one workflow above is recommended for normal scientific use.

Individual stages can also be inspected explicitly:

    runner = BayMorphRunner(joint_config_path)

    runner.prepare_image()
    runner.select_psf_candidates()
    runner.build_psf()

    result = runner.fit()

    runner.corner()
    runner.residual_maps()
    profile_bands = runner.radial_profile()

The global posterior and Bayesian evidence describe the complete joint
probabilistic model. Detector-space model images and residual diagnostics
remain observation-specific.

In sky-frame inference, BayMorph maps sky-coordinate parameters through the
WCS of each observation rather than requiring the images to be resampled onto
a common detector grid. This template keeps sky position independent between
observations; sharing it requires validated astrometric registration.

Changing which parameters are shared changes the scientific model.
Consequently, it also changes the interpretation of the Bayesian evidence.
